# Attention U-Net implementation on 128x128 px tiles

## Setup and Imports

In [ ]:
!pip install rasterio

In [ ]:
import os
import sys
import json
import numpy as np
import tensorflow as tf
import rasterio
import warnings
from zoneinfo import ZoneInfo
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
from tensorflow.keras import layers, models, callbacks, mixed_precision
from tensorflow.keras import backend as K

print(f"TensorFlow version: {tf.__version__}")
print(f"Python version: {sys.version}")

## GPU configuration

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        gpu_info = !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
        print("GPU Information:")
        print(gpu_info[0])

        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        print(f"\nMixed precision policy: {policy.name}")
        print("Compute dtype:", policy.compute_dtype)
        print("Variable dtype:", policy.variable_dtype)

    except RuntimeError as e:
        print(f"GPU setup error: {e}")
else:
    print("No GPUs found")

## Pull data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists('/content/Preprocessed-128'):
    !cp /content/drive/MyDrive/Preprocessed-128.zip /content/
    !unzip -q /content/Preprocessed-128.zip -d /content/

# Define paths
BASE_PATH = Path('/content/Preprocessed-128')
TRAIN_SAR = BASE_PATH / 'train' / 'sar'
TRAIN_FLOOD = BASE_PATH / 'train' / 'flood'
VAL_SAR = BASE_PATH / 'val' / 'sar'
VAL_FLOOD = BASE_PATH / 'val' / 'flood'
TEST_SAR = BASE_PATH / 'test' / 'sar'
TEST_FLOOD = BASE_PATH / 'test' / 'flood'
METADATA_PATH = BASE_PATH / 'metadata'

# Verify paths exist
print("Checking data paths...")
for path_name, path in [
    ("Base", BASE_PATH),
    ("Train SAR", TRAIN_SAR),
    ("Train Flood", TRAIN_FLOOD),
    ("Val SAR", VAL_SAR),
    ("Val Flood", VAL_FLOOD),
    ("Test SAR", TEST_SAR),
    ("Test Flood", TEST_FLOOD),
    ("Metadata", METADATA_PATH)
]:
    if path.exists():
        if path.is_dir() and path_name != "Base" and path_name != "Metadata":
            file_count = len(list(path.glob('*.tif')))
            print(f"{path_name}: {file_count} files")
        else:
            print(f"{path_name}: exists")
    else:
        print(f"{path_name}: not found at {path}")

norm_stats_path = METADATA_PATH / 'normalization_stats.json'
if norm_stats_path.exists():
    with open(norm_stats_path, 'r') as f:
        norm_stats = json.load(f)

## Data loading configuration

In [ ]:
IMG_HEIGHT = 128
IMG_WIDTH = 128
CHANNELS = 3
BATCH_SIZE = 32
BUFFER_SIZE = 1000
AUTOTUNE = tf.data.AUTOTUNE

print(f"Batch size: {BATCH_SIZE}")

## Data pipeline functions

In [ ]:
warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)

def load_image_pair(sar_path, flood_path):
    
    with rasterio.open(sar_path.numpy().decode()) as src:
        sar_data = src.read().transpose(1, 2, 0).astype(np.float32)

    with rasterio.open(flood_path.numpy().decode()) as src:
        flood_data = src.read(1).astype(np.float32)
        flood_data = np.expand_dims(flood_data, axis=-1)

    return sar_data, flood_data

def tf_load_image_pair(sar_path, flood_path):

    sar_data, flood_data = tf.py_function(
        load_image_pair,
        [sar_path, flood_path],
        [tf.float32, tf.float32]
    )

    sar_data.set_shape([IMG_HEIGHT, IMG_WIDTH, CHANNELS])
    flood_data.set_shape([IMG_HEIGHT, IMG_WIDTH, 1])

    return sar_data, flood_data

def augment(sar, mask):
    
    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        sar = tf.image.flip_left_right(sar)
        mask = tf.image.flip_left_right(mask)

    # Random vertical flip
    if tf.random.uniform(()) > 0.5:
        sar = tf.image.flip_up_down(sar)
        mask = tf.image.flip_up_down(mask)

    # Random 90 degree rotations
    k = tf.random.uniform((), maxval=4, dtype=tf.int32)
    sar = tf.image.rot90(sar, k)
    mask = tf.image.rot90(mask, k)

    # Random brightness adjustment for SAR only
    sar = tf.image.random_brightness(sar, 0.1)

    return sar, mask

def create_dataset(sar_dir, flood_dir, training=False, batch_size=BATCH_SIZE):

    sar_files = list(Path(sar_dir).glob('*.tif'))

    flood_map = {}
    for f in Path(flood_dir).glob('*.tif'):
        key = f.stem.replace('_flood_prep', '_prep')
        flood_map[key] = str(f)

    matched_pairs = []

    for sar_file in sar_files:
        sar_stem = sar_file.stem
        if sar_stem in flood_map:
            matched_pairs.append((str(sar_file), flood_map[sar_stem]))

    sar_paths = [pair[0] for pair in matched_pairs]
    flood_paths = [pair[1] for pair in matched_pairs]

    dataset = tf.data.Dataset.from_tensor_slices((sar_paths, flood_paths))

    if training:
        dataset = dataset.shuffle(buffer_size=BUFFER_SIZE, seed=42)

    dataset = dataset.map(tf_load_image_pair, num_parallel_calls=AUTOTUNE)

    if training:
        dataset = dataset.map(augment, num_parallel_calls=AUTOTUNE)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(AUTOTUNE)

    return dataset, len(matched_pairs)

## Create datasets

In [ ]:
train_dataset, train_size = create_dataset(TRAIN_SAR, TRAIN_FLOOD, training=True)
val_dataset, val_size = create_dataset(VAL_SAR, VAL_FLOOD, training=False)
test_dataset, test_size = create_dataset(TEST_SAR, TEST_FLOOD, training=False)

print(f"Train: {train_size} images ({train_size // BATCH_SIZE} batches)")
print(f"Validation: {val_size} images ({val_size // BATCH_SIZE} batches)")
print(f"Test: {test_size} images ({test_size // BATCH_SIZE} batches)")

for i, (sar, mask) in enumerate(train_dataset.take(1)):
    print(f"SAR batch shape: {sar.shape}, dtype: {sar.dtype}")
    print(f"Mask batch shape: {mask.shape}, dtype: {mask.dtype}")
    print(f"SAR value range: [{tf.reduce_min(sar):.3f}, {tf.reduce_max(sar):.3f}]")
    print(f"Mask unique values: {tf.unique(tf.reshape(mask, [-1]))[0].numpy()}")

## U-Net model architecture

In [ ]:
def conv_block(inputs, filters, kernel_size=3, dropout_rate=0.1):
    # Convolutional block with batch normalization and dropout
    x = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer='he_normal')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv2D(filters, kernel_size, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    return x

def encoder_block(inputs, filters, pool_size=2, dropout_rate=0.1):
    # Encoder block with convolution and pooling
    conv = conv_block(inputs, filters, dropout_rate=dropout_rate)
    pool = layers.MaxPooling2D(pool_size)(conv)
    return conv, pool

def attention_gate(F_g, F_l, filters):
    """Attention mechanism"""
    # Gating signal processing
    W_g = layers.Conv2D(filters, 1, padding='same')(F_g)
    W_g = layers.BatchNormalization()(W_g)
    
    # Feature map processing  
    W_x = layers.Conv2D(filters, 1, padding='same')(F_l)
    W_x = layers.BatchNormalization()(W_x)
    
    # Attention computation
    psi = layers.Add()([W_g, W_x])
    psi = layers.Activation('relu')(psi)
    psi = layers.Conv2D(1, 1, padding='same')(psi)
    psi = layers.BatchNormalization()(psi)
    psi = layers.Activation('sigmoid')(psi)
    
    # Apply attention
    result = layers.Multiply()([F_l, psi])
    return result

def attention_decoder_block(inputs, skip_features, filters, dropout_rate=0.1):
    """Decoder block with attention mechanism"""
    # Upsample
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(inputs)
    
    # Apply attention to skip connection
    skip_attended = attention_gate(x, skip_features, filters // 2)
    
    # Concatenate and process
    x = layers.concatenate([x, skip_attended])
    x = conv_block(x, filters, dropout_rate=dropout_rate)
    return x

def build_attention_unet(input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), dropout_rate=0.1):
    inputs = layers.Input(shape=input_shape)
    
    # Encoder
    conv1, pool1 = encoder_block(inputs, 64, dropout_rate=dropout_rate)
    conv2, pool2 = encoder_block(pool1, 128, dropout_rate=dropout_rate)
    conv3, pool3 = encoder_block(pool2, 256, dropout_rate=dropout_rate)
    conv4, pool4 = encoder_block(pool3, 512, dropout_rate=dropout_rate)
    
    # Bridge
    bridge = conv_block(pool4, 1024, dropout_rate=dropout_rate)
    
    # Decoder with attention
    dec4 = attention_decoder_block(bridge, conv4, 512, dropout_rate=dropout_rate)
    dec3 = attention_decoder_block(dec4, conv3, 256, dropout_rate=dropout_rate)
    dec2 = attention_decoder_block(dec3, conv2, 128, dropout_rate=dropout_rate)
    dec1 = attention_decoder_block(dec2, conv1, 64, dropout_rate=dropout_rate)
    
    # Output
    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32')(dec1)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

model = build_attention_unet(dropout_rate=0.1)
print(f"Total parameters: {model.count_params():,}")

## Loss Functions and Metrics

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred)
    union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)

    dice = (2.0 * intersection + smooth) / (union + smooth)
    return dice

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

# Custom metrics
class SegmentationMetrics(tf.keras.metrics.Metric):
    def __init__(self, name='segmentation_metrics', **kwargs):
        super().__init__(name=name, **kwargs)
        self.true_positives = self.add_weight(name='tp', initializer='zeros')
        self.false_positives = self.add_weight(name='fp', initializer='zeros')
        self.false_negatives = self.add_weight(name='fn', initializer='zeros')
        self.true_negatives = self.add_weight(name='tn', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true > 0.5, tf.float32)
        y_pred = tf.cast(y_pred > 0.5, tf.float32)

        self.true_positives.assign_add(tf.reduce_sum(y_true * y_pred))
        self.false_positives.assign_add(tf.reduce_sum((1 - y_true) * y_pred))
        self.false_negatives.assign_add(tf.reduce_sum(y_true * (1 - y_pred)))
        self.true_negatives.assign_add(tf.reduce_sum((1 - y_true) * (1 - y_pred)))

    def result(self):
        precision = self.true_positives / (self.true_positives + self.false_positives + K.epsilon())
        recall = self.true_positives / (self.true_positives + self.false_negatives + K.epsilon())
        return {'precision': precision, 'recall': recall}

    def reset_state(self):
        self.true_positives.assign(0.)
        self.false_positives.assign(0.)
        self.false_negatives.assign(0.)
        self.true_negatives.assign(0.)

## Compile model

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
optimizer = mixed_precision.LossScaleOptimizer(optimizer)

model.compile(
    optimizer=optimizer,
    loss=combined_loss,
    metrics=[
        dice_coefficient,
        tf.keras.metrics.BinaryAccuracy(name='pixel_accuracy'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

print("Model compiled successfully")

## Training configuration

In [ ]:
checkpoint_dir = Path('/content/drive/MyDrive/unet_128_checkpoints')
checkpoint_dir.mkdir(exist_ok=True)

callbacks_list = [
    callbacks.ModelCheckpoint(
        filepath=str(checkpoint_dir / 'best_model.h5'),
        monitor='val_dice_coefficient',
        mode='max',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),

    callbacks.EarlyStopping(
        monitor='val_dice_coefficient',
        mode='max',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),

    callbacks.CSVLogger(
        str(checkpoint_dir / 'training_log.csv'),
        append=False
    )
]

EPOCHS = 80
STEPS_PER_EPOCH = train_size // BATCH_SIZE
VALIDATION_STEPS = val_size // BATCH_SIZE

print(f"\nTraining configuration:")
print(f"Epochs: {EPOCHS}")
print(f"Steps per epoch: {STEPS_PER_EPOCH}")
print(f"Validation steps: {VALIDATION_STEPS}")
print(f"Total training samples per epoch: {STEPS_PER_EPOCH * BATCH_SIZE}")

## Train model

In [ ]:
print(f"\nStarting training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_dataset,
    validation_steps=VALIDATION_STEPS,
    callbacks=callbacks_list,
    verbose=1
)

print(f"\nTraining completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Plot training history

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(history.history['loss'], label='Train')
axes[0, 0].plot(history.history['val_loss'], label='Validation')
axes[0, 0].set_title('Model Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history.history['dice_coefficient'], label='Train')
axes[0, 1].plot(history.history['val_dice_coefficient'], label='Validation')
axes[0, 1].set_title('Dice Coefficient')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Dice')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(history.history['pixel_accuracy'], label='Train')
axes[1, 0].plot(history.history['val_pixel_accuracy'], label='Validation')
axes[1, 0].set_title('Pixel Accuracy')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history.history['precision'], label='Train Precision')
axes[1, 1].plot(history.history['recall'], label='Train Recall')
axes[1, 1].plot(history.history['val_precision'], label='Val Precision')
axes[1, 1].plot(history.history['val_recall'], label='Val Recall')
axes[1, 1].set_title('Precision and Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(checkpoint_dir / 'training_history.png', dpi=300, bbox_inches='tight')
plt.show()

## Evaluate model on test set

In [ ]:
best_model_path = checkpoint_dir / 'best_model.h5'
if best_model_path.exists():
    print("Loading best model...")
    model = tf.keras.models.load_model(
        best_model_path,
        custom_objects={
            'combined_loss': combined_loss,
            'dice_coefficient': dice_coefficient
        }
    )
else:
    print("Using final model (best model checkpoint not found)")

test_results = model.evaluate(
    test_dataset,
    steps=test_size // BATCH_SIZE,
    verbose=1
)

metric_names = ['Loss', 'Dice Coefficient', 'Pixel Accuracy', 'Precision', 'Recall']
print("\nTest Set Results:")
for name, value in zip(metric_names, test_results):
    print(f"{name}: {value:.4f}")

## Save final results

In [ ]:
results_summary = {
    'training_completed': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'model_parameters': model.count_params(),
    'training_samples': train_size,
    'validation_samples': val_size,
    'test_samples': test_size,
    'batch_size': BATCH_SIZE,
    'epochs_trained': len(history.history['loss']),
    'test_results': {
        'loss': float(test_results[0]),
        'dice_coefficient': float(test_results[1]),
        'pixel_accuracy': float(test_results[2]),
        'precision': float(test_results[3]),
        'recall': float(test_results[4]),
        'f1_score': float(f1_score) if 'f1_score' in locals() else None
    },
    'best_validation_dice': float(max(history.history['val_dice_coefficient'])),
    'best_validation_accuracy': float(max(history.history['val_pixel_accuracy']))
}

results_path = checkpoint_dir / 'training_results.json'
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\nResults saved to: {results_path}")
print("\nTraining pipeline completed")